In [3]:
%pip -q install "qiskit==2.2.3" "numpy==2.3.5" "scipy==1.16.3"

In [4]:
import time
import importlib.metadata as metadata
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('versions:', {k: metadata.version(k) for k in ['qiskit','numpy','scipy']})

def solve(names, cost, pairs, equalities, forbidden, feasible, describe,
          penalty=100.0, p=2, maxiter=160, shots=4096):
    n = len(names)
    a = np.array(cost, dtype=float)
    b = dict(pairs)
    constant = 0.0
    for indices, target in equalities:
        constant += penalty * target**2
        for i in indices:
            a[i] += penalty * (1 - 2 * target)
        for k, i in enumerate(indices):
            for j in indices[k+1:]:
                key = tuple(sorted((i,j)))
                b[key] = b.get(key,0.0) + 2*penalty
    for i,j in forbidden:
        key = tuple(sorted((i,j)))
        b[key] = b.get(key,0.0) + penalty

    # x_i=(1-Z_i)/2 치환을 적용한 Ising 계수
    h = -a/2
    offset = constant + a.sum()/2 + sum(b.values())/4
    zz = {key:value/4 for key,value in b.items()}
    for (i,j),value in b.items():
        h[i] -= value/4
        h[j] -= value/4
    bits = ((np.arange(2**n)[:,None] >> np.arange(n)) & 1).astype(int)
    z = 1 - 2*bits
    energy = constant + bits@a
    original = bits@np.array(cost,dtype=float)
    ising = offset + z@h
    for (i,j),value in b.items():
        energy += value*bits[:,i]*bits[:,j]
        ising += value*z[:,i]*z[:,j]/4
    for (i,j),value in pairs.items():
        original += value*bits[:,i]*bits[:,j]
    valid = np.array([feasible(x) for x in bits],dtype=bool)
    violation = np.zeros(2**n)
    for indices,target in equalities:
        violation += (bits[:,indices].sum(axis=1)-target)**2
    for i,j in forbidden:
        violation += bits[:,i]*bits[:,j]
    assert np.array_equal(valid,np.isclose(violation,0)), '원래 제약과 QUBO 제약을 대조하여 확인하세요.'
    assert np.allclose(energy,original+penalty*violation)
    assert valid.any() and np.allclose(energy,ising)
    assert np.allclose(energy[valid],original[valid])
    exact = original[valid].min()
    assert np.all(energy[~valid] > exact), '벌점 크기를 검토하세요.'
    optimum = valid & np.isclose(original,exact)
    scale = max(1.0,float(np.max(np.abs(energy))))

    def circuit(theta):
        qc = QuantumCircuit(n)
        qc.h(range(n))
        for layer in range(p):
            gamma = theta[2*layer]     # γ(감마): 비용 회로 각도
            beta = theta[2*layer+1]    # β(베타): 믹서 회로 각도
            for i,value in enumerate(h):
                qc.rz(2*gamma*value/scale,i)
            for (i,j),value in zz.items():
                qc.rzz(2*gamma*value/scale,i,j)
            for i in range(n):
                qc.rx(2*beta,i)
        return qc

    history = []
    def objective(theta):
        probability = Statevector.from_instruction(circuit(theta)).probabilities()
        value = float(probability@energy/scale)
        history.append(value)
        return value

    start = time.perf_counter()
    runs = []
    for gamma0,beta0 in [(0.8,0.3),(1.7,0.7)]:
        initial = np.array([gamma0,beta0]*p)
        runs.append(minimize(objective,initial,method='COBYLA',
                             options={'maxiter':maxiter,'rhobeg':0.5,'tol':1e-5}))
    result = min(runs,key=lambda r:r.fun)
    qc = circuit(result.x)
    state = Statevector.from_instruction(qc)
    probability = state.probabilities()
    state.seed(42)
    counts = state.sample_counts(shots)
    observed = np.array([counts.get(format(i,f'0{n}b'),0) for i in range(2**n)])
    candidates = np.flatnonzero(valid & (observed>0))
    print('큐비트 수:',n,'층수:',p,'측정 횟수:',shots)
    print('최적화 종료:',[(r.success,str(r.message)) for r in runs])
    print('총 목적함수 평가:',len(history),'실행시간(초):',round(time.perf_counter()-start,3))
    print('QUBO 계수:',constant,a.tolist(),b)
    print('Ising 계수:',offset,h.tolist(),zz)
    print('정규화 계수:',scale,'최적 각도:',result.x)
    print('정확 최적값:',exact)
    print('기대 QUBO 에너지:',float(probability@energy))
    print('유효해 확률:',float(probability[valid].sum()),'최적해 확률:',float(probability[optimum].sum()))
    print('유효해 관측 비율:',observed[valid].sum()/shots,'최적해 관측 비율:',observed[optimum].sum()/shots)
    if not len(candidates):
        print('유효한 해가 관측되지 않았습니다. 각도·층수·평가 횟수를 조정하세요.')
        return
    best = min(candidates,key=lambda i:(original[i],-observed[i]))
    mode = int(np.argmax(observed))
    print('최빈 비트열:',format(mode,f'0{n}b'),'제약 충족:',bool(valid[mode]))
    print('선택 비트(변수 순서):',bits[best].tolist())
    print('선택 항목:',[names[i] for i in range(n) if bits[best,i]])
    print('선택 목적값:',original[best],'정확해 차이:',original[best]-exact)
    describe(bits[best])
    print('목적함수 평가 이력(시작/종료):',history[0],history[-1])
    assert feasible(bits[best]) and original[best] >= exact-1e-8
    assert np.isclose(probability.sum(),1) and observed.sum()==shots
    print('회로 깊이:',qc.depth(),'게이트 수:',dict(qc.count_ops()))


names = ['A_일반','A_긴급','B_일반','B_긴급']
cost = [8,13,7,12]
pairs = {(0,2):6}
equalities = [([0,1],1),([2,3],1)]
forbidden = []
def feasible(x):
    return x[0]+x[1]==1 and x[2]+x[3]==1
def describe(x):
    print('A부품 100개 확보:',bool(x[0]+x[1]==1))
    print('B부품 80개 확보:',bool(x[2]+x[3]==1))
    print('발주 평가비용(만원):',np.dot(cost,x)+sum(value*x[i]*x[j] for (i,j),value in pairs.items()))
solve(names,cost,pairs,equalities,forbidden,feasible,describe)

versions: {'qiskit': '2.2.3', 'numpy': '2.3.5', 'scipy': '1.16.3'}
큐비트 수: 4 층수: 2 측정 횟수: 4096
최적화 종료: [(False, 'Return from COBYLA because the objective function has been evaluated MAXFUN times.'), (False, 'Return from COBYLA because the objective function has been evaluated MAXFUN times.')]
총 목적함수 평가: 320 실행시간(초): 1.96
QUBO 계수: 200.0 [-92.0, -87.0, -93.0, -88.0] {(0, 2): 6, (0, 1): 200.0, (2, 3): 200.0}
Ising 계수: 121.5 [-5.5, -6.5, -5.0, -6.0] {(0, 2): 1.5, (0, 1): 50.0, (2, 3): 50.0}
정규화 계수: 246.0 최적 각도: [ 2.08688063  1.56950376  1.66998137 -0.39178281]
정확 최적값: 20.0
기대 QUBO 에너지: 21.697114426650614
유효해 확률: 0.9982701123724991 최적해 확률: 0.49965459524911066
유효해 관측 비율: 0.997802734375 최적해 관측 비율: 0.489990234375
최빈 비트열: 0101 제약 충족: True
선택 비트(변수 순서): [0, 1, 1, 0]
선택 항목: ['A_긴급', 'B_일반']
선택 목적값: 20.0 정확해 차이: 0.0
A부품 100개 확보: True
B부품 80개 확보: True
발주 평가비용(만원): 20
목적함수 평가 이력(시작/종료): 0.6999028527269339 0.1017730721420648
회로 깊이: 9 게이트 수: {'rz': 8, 'rx': 8, 'rzz': 6, 'h': 4}
